In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return x @ pruned_weights.T + self.bias

    def get_gate_values(self):
        return torch.sigmoid(self.gate_scores)

In [3]:
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(32*32*3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def get_all_gates(self):
        gates = []
        for layer in [self.fc1, self.fc2, self.fc3]:
            gates.append(layer.get_gate_values().view(-1))
        return torch.cat(gates)

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

print("Data loaded")

100%|██████████| 170M/170M [00:11<00:00, 15.1MB/s]


Data loaded


In [5]:
def train_model(lambda_val, epochs=3):
    model = PrunableNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            ce_loss = criterion(outputs, labels)

            gates = model.get_all_gates()
            sparsity_loss = torch.sum(gates)

            loss = ce_loss + lambda_val * sparsity_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Lambda {lambda_val} | Epoch {epoch+1} done")

    return model

In [6]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    gates = model.get_all_gates().cpu().numpy()
    sparsity = np.mean(gates < 1e-2) * 100

    return accuracy, sparsity, gates

In [8]:
lambdas = [1e-5, 1e-4, 1e-3]
results = []

best_model = None
best_acc = 0
best_gates = None

for lam in lambdas:
    print(f"\nRunning lambda = {lam}")

    model = train_model(lam, epochs=2)  # faster + safer

    acc, sparsity, gates = evaluate(model)

    results.append((lam, acc, sparsity))

    if acc > best_acc or best_gates is None:
        best_acc = acc
        best_model = model
        best_gates = gates

    print(f"Lambda: {lam} | Accuracy: {acc:.2f}% | Sparsity: {sparsity:.2f}%")


Running lambda = 1e-05
Lambda 1e-05 | Epoch 1 done
Lambda 1e-05 | Epoch 2 done


RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [ ]:
print("\nFinal Results:")
print("Lambda\tAccuracy\tSparsity")
for r in results:
    print(f"{r[0]}\t{r[1]:.2f}%\t\t{r[2]:.2f}%")

In [ ]:
plt.hist(best_gates, bins=50)
plt.title("Gate Value Distribution (Best Model)")
plt.xlabel("Gate Values")
plt.ylabel("Frequency")
plt.show()